# RAILGUN+ — LaCAM Data Generation (Kaggle)

Generates expert trajectories with **LaCAM** (near-SoC-optimal) instead of PIBT.
This is the key change that lets the learned policy *beat* pure PIBT: the net
distills an expensive optimal solver that's too slow to run at inference.

**One-time setup:** build LaCAM once, save the binary as a Kaggle dataset, then
load it here. After that, no compilation.

## 0. (ONE TIME) Build LaCAM and save as a dataset
Run this ONCE in a throwaway notebook, then save `/kaggle/working/lacam/build/main` as a Kaggle Dataset. Skip if you already have the dataset attached.

In [ ]:
# !git clone https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
# !bash /kaggle/working/railgun-plus/scripts/build_lacam.sh
# -> then download the printed binary path and create a Kaggle Dataset from it.

## 1. Setup: clone repo, install pogema, locate LaCAM binary

In [ ]:
import os, sys
!git clone -q https://github.com/tay805/railgun-plus.git /kaggle/working/railgun-plus
!pip install -q pogema
sys.path.insert(0,'/kaggle/working/railgun-plus/src')

# Path to your prebuilt LaCAM binary (from the Kaggle dataset you attached).
# EDIT THIS to match your attached dataset path:
LACAM_BIN='/kaggle/input/lacam-binary/main'
assert os.path.exists(LACAM_BIN), f'LaCAM binary not found at {LACAM_BIN} - attach your lacam dataset and fix this path'
!chmod +x {LACAM_BIN}
print('LaCAM binary ready:', LACAM_BIN)

## 2. Output dir (persists in /kaggle/working; download or save as dataset after)

In [ ]:
DATA_DIR='/kaggle/working/data_lacam/shards'
os.makedirs(DATA_DIR, exist_ok=True)
print(DATA_DIR)

## 3. Smoke test: generate a few LaCAM instances
Verifies the binary + parser work BEFORE the long run. If the parser can't read your LaCAM build's output, fix `parse_lacam_solution` in lacam_expert.py (see its docstring).

In [ ]:
!cd /kaggle/working/railgun-plus && PYTHONPATH=src python scripts/generate_data.py \
    --out {DATA_DIR} --expert lacam --lacam-bin {LACAM_BIN} \
    --instances 5 --map-size 32 --density 0.2 --agents 32 --seed 0 --batch 5 --lacam-time-ms 10000

## 4. Full generation across agent counts (resumable)
If Kaggle disconnects, just re-run this cell — it resumes per agent count.

In [ ]:
for agents in [16,32,64,96]:
    !cd /kaggle/working/railgun-plus && PYTHONPATH=src python scripts/generate_data.py \
        --out {DATA_DIR} --expert lacam --lacam-bin {LACAM_BIN} \
        --instances 200 --map-size 32 --density 0.2 --agents {agents} --seed {agents} \
        --batch 20 --lacam-time-ms 10000

## 5. Save the generated data as a Kaggle Dataset
So training (a separate notebook) can load it without regenerating. Use Kaggle's 'Save Version' or create a dataset from /kaggle/working/data_lacam.

In [ ]:
import glob
print('shards:', len(glob.glob(f'{DATA_DIR}/*.npz')))
print('Now: File -> save, or create a dataset from', DATA_DIR)